# CNIBP One-Click (VSCode + Colab)\n在 VSCode 的 Colab 插件连接远程 runtime 后，从上到下 `Run All`。

In [ ]:
# 只需要改这3项
GIT_REPO = 'https://github.com/67vmg9wrfn-beep/Lab.git'
GIT_BRANCH = 'main'
PROJECT_SUBDIR = '06_experiments/cnibp/repro_ppg_bp'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
if os.path.exists('/content/repo_src'):
    shutil.rmtree('/content/repo_src')
subprocess.run(['git','clone','--depth','1','--branch', GIT_BRANCH, GIT_REPO, '/content/repo_src'], check=True)
print('git clone done')


In [ ]:
import os
PROJECT_ROOT = f'/content/repo_src/{PROJECT_SUBDIR}'
print('PROJECT_ROOT=', PROJECT_ROOT)
assert os.path.exists(PROJECT_ROOT), f'Path not found: {PROJECT_ROOT}'

In [ ]:
from pathlib import Path
import re

BASE = Path('/content/drive/MyDrive')
if not BASE.exists():
    raise FileNotFoundError('Drive not mounted at /content/drive/MyDrive')

mat_files = [f for f in BASE.rglob('*.mat') if '/__MACOSX/' not in str(f)]
print(f'[INFO] total .mat files found (without __MACOSX): {len(mat_files)}')

pat = re.compile(r'^part_([0-9]+)\.mat$', re.IGNORECASE)
by_dir = {}
for f in mat_files:
    m = pat.match(f.name)
    if not m:
        continue
    d = str(f.parent)
    by_dir.setdefault(d, {})[int(m.group(1))] = f.name

if not by_dir:
    raise FileNotFoundError('No Part_*.mat files found under /content/drive/MyDrive')

# Prefer directory with most parts, and containing kachuee/raw_mat
dirs = sorted(by_dir.keys(), key=lambda d: (len(by_dir[d]), 'kachuee' in d.lower() and 'raw_mat' in d.lower()), reverse=True)
DATA_ROOT = dirs[0]
part_map = by_dir[DATA_ROOT]
part_ids = sorted(part_map.keys())
PARTS = [part_map[i] for i in part_ids]

print('[OK] DATA_ROOT =', DATA_ROOT)
print('[OK] detected parts =', PARTS)
if 0 not in part_ids:
    print('[WARN] Part_0.mat not found; running with available parts only.')
if len(PARTS) < 2:
    raise RuntimeError('Too few parts to train reliably. Need at least 2 Part_*.mat files.')


In [ ]:
import subprocess
subprocess.run(['python','-m','pip','install','-r', f'{PROJECT_ROOT}/requirements_colab.txt'], check=True)
subprocess.run(['python','-m','pip','install','-e', PROJECT_ROOT], check=True)
print('dependencies installed')


In [ ]:
import os, glob, subprocess
OUT_ROOT='/content/drive/MyDrive/cnibp_repro_outputs'
os.makedirs(OUT_ROOT, exist_ok=True)
log_file=f'{OUT_ROOT}/last_run.log'
parts_arg=','.join(PARTS)

# Auto-reuse latest preprocessing artifacts if available
reuse_candidates=[]
for d in sorted(glob.glob(f'{OUT_ROOT}/run_*')):
    m=f'{d}/preprocess/preprocessed_manifest.json'
    s=f'{d}/preprocess/segments_meta.csv'
    if os.path.exists(m) and os.path.exists(s):
        reuse_candidates.append(d)
reuse_from = reuse_candidates[-1] if reuse_candidates else ''

cmd=[
    'python','-m','cnibp_repro.run_repro',
    '--drive_root', DATA_ROOT,
    '--parts', parts_arg,
    '--config',f'{PROJECT_ROOT}/configs/paper_repro.json',
    '--output_root',OUT_ROOT
]
if reuse_from:
    cmd += ['--reuse_from_run_dir', reuse_from]

print('[INFO] run parts =', parts_arg)
print('[INFO] reuse_from_run_dir =', reuse_from if reuse_from else '(none)')
with open(log_file, 'w', encoding='utf-8') as f:
    proc=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        f.write(line)
    code=proc.wait()
if code != 0:
    raise RuntimeError(f'run_repro failed, see {log_file}')
print(f'log saved: {log_file}')


In [ ]:
# 失败时你只需要看这个输出路径
print('/content/drive/MyDrive/cnibp_repro_outputs/last_run.log')